# 01 — Feature Extraction

**Project:** VBX — Virtual Molecular Biopsy · Manipal Hackathon 2026  
**Track:** ML · Layer 1 Data  
**Author:** Aman Mishra  

## Datasets used (add via Kaggle → Add Data before running)
| Kaggle ref | Role |
|---|---|
| `usmansadiqcs/ucsf-pdgm-v3-dataset` | Clinical CSV (`UCSF-PDGM-metadata.csv`) |
| `rukiyeaydn/ucsf-pdgm-filtered` | NIfTI volumes: T1, T1c, T2, FLAIR, tumor segmentation |

Both datasets come from UCSF-PDGM (TCIA, CC BY 4.0). No separate crosswalk is required;  
the patient IDs in the clinical CSV match the directory names in the volume dataset.

## Segmentation labels (UCSF-PDGM, same as BraTS standard)
- Label **1** = Necrotic Core (NCR)
- Label **2** = Peritumoral Edema (ED)
- Label **4** = Enhancing Tumor (ET)
- Label **3** = unused

## Leakage policy
Inputs: **age, sex only.**  
Targets: IDH status, 1p/19q co-deletion, MGMT methylation, WHO grade.  
Never used as inputs: WHO grade, pathology diagnosis, extent of resection, treatment,  
survival, vital status, MGMT index, or any post-operative variable.

## Column naming convention
- `{sequence}__{region}__{featureclass}__{feature}` — radiomics (e.g. `t1ce__et__glcm__Contrast`)
- `shape__{region}__shape__{feature}` — shape (extracted once per region)
- `desc__{name}` — interpretable descriptors (e.g. `desc__total_volume_mm3`)
- `age`, `sex` — clinical inputs
- `idh`, `codeletion_1p19q`, `mgmt`, `who_grade` — targets
- `patient_id`, `split`, `fold` — admin
- `missing__{region}` — flag columns (1 if region mask is empty)

## 0. Install pinned dependencies

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# scipy, skimage, SimpleITK are all pre-installed on the Kaggle image.
# Only extra deps needed:
_pip('pandera==0.20.4', 'pyarrow', 'tqdm')
print('Installs done')


In [ ]:
import hashlib, json, os, re, warnings, pickle, tempfile
import numpy as np
import pandas as pd
import SimpleITK as sitk
import pandera as pa
from pandera import Column, DataFrameSchema, Check
from pathlib import Path
from tqdm.auto import tqdm
from scipy import stats, ndimage
from skimage.feature import graycomatrix, graycoprops
from skimage import measure

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

import sklearn
print(f'numpy       {np.__version__}')
print(f'SimpleITK   {sitk.Version.VersionString()}')
print(f'scipy       {__import__("scipy").__version__}')
print(f'skimage     {__import__("skimage").__version__}')
print(f'pandera     {pa.__version__}')


## 1. Radiomics config and provenance record

## 2. Paths

In [ ]:
# Dataset mount points on Kaggle
CLINICAL_CSV  = Path('/kaggle/input/ucsf-pdgm-v3-dataset/UCSF-PDGM-metadata.csv')
VOL_ROOT      = Path('/kaggle/input/ucsf-pdgm-filtered/ucsf')
OUT_DIR       = Path('/kaggle/working')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Clinical CSV exists : {CLINICAL_CSV.exists()}')
print(f'Volume root exists  : {VOL_ROOT.exists()}')
if VOL_ROOT.exists():
    patients_on_disk = sorted([d.name for d in VOL_ROOT.iterdir() if d.is_dir()])
    print(f'Patient dirs found  : {len(patients_on_disk)}')
    print(f'Example dirs        : {patients_on_disk[:3]}')

## 3. Load and inspect clinical CSV

In [ ]:
raw = pd.read_csv(CLINICAL_CSV)
print(f'Raw CSV shape: {raw.shape}')
print('Columns:', list(raw.columns))
raw.head(3)

In [ ]:
# Print unique values for molecular marker columns to confirm encoding format
for col in raw.columns:
    if any(kw in col.upper() for kw in ['IDH', 'CODEL', '1P', 'MGMT', 'GRADE', 'SEX', 'GENDER']):
        print(f'{col}: {raw[col].unique()[:10]}')

## 4. Select columns and canonical rename

In [ ]:
def _find_col(df, patterns):
    """Return the first column matching any pattern (case-insensitive substring)."""
    for pat in patterns:
        matches = [c for c in df.columns if pat.lower() in c.lower()]
        if matches:
            return matches[0]
    return None

ID_COL    = _find_col(raw, ['ID', 'Patient', 'Subject', 'Case'])
AGE_COL   = _find_col(raw, ['Age', 'AGE', 'age_at'])
SEX_COL   = _find_col(raw, ['Sex', 'Gender'])
IDH_COL   = _find_col(raw, ['IDH_status', 'IDH status', 'IDH'])
COD_COL   = _find_col(raw, ['1p19q', '1p/19q', 'codeletion', 'co-deletion', 'codel'])
MGMT_COL  = _find_col(raw, ['MGMT_status', 'MGMT status', 'MGMT'])
GRADE_COL = _find_col(raw, ['WHO_Grade', 'WHO grade', 'Grade'])

for name, val in [('ID',ID_COL),('Age',AGE_COL),('Sex',SEX_COL),
                  ('IDH',IDH_COL),('1p19q',COD_COL),('MGMT',MGMT_COL),('Grade',GRADE_COL)]:
    print(f'  {name:8s} → {val}')

In [ ]:
col_map = {ID_COL: 'patient_id_raw'}
if AGE_COL:   col_map[AGE_COL]   = 'age_raw'
if SEX_COL:   col_map[SEX_COL]   = 'sex_raw'
if IDH_COL:   col_map[IDH_COL]   = 'idh_raw'
if COD_COL:   col_map[COD_COL]   = 'codeletion_raw'
if MGMT_COL:  col_map[MGMT_COL]  = 'mgmt_raw'
if GRADE_COL: col_map[GRADE_COL] = 'grade_raw'

clinical = raw[list(col_map)].rename(columns=col_map).copy()
print(f'clinical shape: {clinical.shape}')
clinical.head(3)

## 5. Deduplicate follow-up scans

UCSF-PDGM v3 contains follow-up entries with IDs like `UCSF-PDGM-0001_FU007d`.  
**Rule:** keep the baseline scan (ID without `_FU`).  
If a patient has only follow-up entries, keep the one with the lowest follow-up day number.  
Reason: follow-up scans reflect post-treatment state, not the pre-operative molecular profile.

In [ ]:
n_before = len(clinical)
clinical['is_followup'] = clinical['patient_id_raw'].str.contains('_FU', na=False)
clinical['base_id'] = clinical['patient_id_raw'].str.replace(r'_FU\d+d$', '', regex=True)

# Extract follow-up day number for tie-breaking (lower = earlier scan)
clinical['fu_days'] = (
    clinical['patient_id_raw']
    .str.extract(r'_FU(\d+)d$')[0]
    .fillna(0)
    .astype(int)
)

n_followup = clinical['is_followup'].sum()

# Sort: baseline (is_followup=False, fu_days=0) first, then by ascending fu_days
clinical_deduped = (
    clinical
    .sort_values(['is_followup', 'fu_days'])
    .drop_duplicates(subset='base_id', keep='first')
    .copy()
)

n_after  = len(clinical_deduped)
n_dropped = n_before - n_after

print(f'Rows in raw CSV             : {n_before}')
print(f'Follow-up rows (_FU)        : {n_followup}')
print(f'Rows after dedup            : {n_after}')
print(f'Rows dropped (dedup rule)   : {n_dropped}')

clinical_deduped['patient_id'] = clinical_deduped['base_id']
clinical_deduped = clinical_deduped.drop(
    columns=['is_followup', 'base_id', 'fu_days', 'patient_id_raw']
)

## 6. Label normalisation

In [ ]:
def _encode_binary(series, pos_vals, neg_vals):
    """Map string labels → 1.0 / 0.0 / NaN. Never imputes."""
    s = series.astype(str).str.strip().str.lower()
    out = pd.Series(np.nan, index=series.index, dtype=float)
    out[s.isin([v.lower() for v in pos_vals])] = 1.0
    out[s.isin([v.lower() for v in neg_vals])] = 0.0
    return out

df = clinical_deduped.copy()

if 'idh_raw' in df:
    df['idh'] = _encode_binary(
        df['idh_raw'],
        pos_vals=['mutant','mut','1','idh mutant','idh-mutant','mutated'],
        neg_vals=['wildtype','wt','0','idh wildtype','idh-wildtype','wild type'],
    )

if 'codeletion_raw' in df:
    df['codeletion_1p19q'] = _encode_binary(
        df['codeletion_raw'],
        pos_vals=['co-deleted','codel','deleted','1','yes','co_deleted'],
        neg_vals=['non-co-deleted','noncodel','non-deleted','0','no','non_co_deleted'],
    )

if 'mgmt_raw' in df:
    df['mgmt'] = _encode_binary(
        df['mgmt_raw'],
        pos_vals=['methylated','meth','1','yes','positive'],
        neg_vals=['unmethylated','unmeth','0','no','negative'],
    )

if 'grade_raw' in df:
    df['who_grade'] = pd.to_numeric(df['grade_raw'], errors='coerce')

if 'sex_raw' in df:
    df['sex'] = _encode_binary(
        df['sex_raw'],
        pos_vals=['m','male','1'],
        neg_vals=['f','female','0'],
    ).astype('Int64')

df['age'] = pd.to_numeric(df.get('age_raw', np.nan), errors='coerce')

# Drop raw columns
df = df.drop(columns=[c for c in df.columns if c.endswith('_raw')], errors='ignore')

print('Label value counts after encoding:')
for col in ['idh','codeletion_1p19q','mgmt','who_grade']:
    if col in df:
        print(f'  {col}: {dict(df[col].value_counts(dropna=False))}')

## 7. Match patients to volumes on disk

In [ ]:
# Build a set of patient IDs present in the volume dataset
# Directory name format: UCSF-PDGM-XXXX_nifti
disk_ids = set()
for d in VOL_ROOT.iterdir():
    if d.is_dir():
        # Strip _nifti suffix to get the patient ID
        pid = d.name.replace('_nifti', '')
        disk_ids.add(pid)

print(f'Patient directories on disk: {len(disk_ids)}')

n_clinical = len(df)
df_matched = df[df['patient_id'].isin(disk_ids)].copy()
n_matched = len(df_matched)
n_missing_vol = n_clinical - n_matched

print(f'Clinical rows (after dedup) : {n_clinical}')
print(f'Matched to volumes          : {n_matched}')
print(f'Missing volume dirs         : {n_missing_vol}')

if n_matched < 100:
    raise RuntimeError(
        f'Only {n_matched} patients matched — expected ~300. '
        'Check patient_id format vs directory names. Stop and report to Tanish.'
    )

## 8. Patient-level train/test split and CV folds

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold

cohort = df_matched.reset_index(drop=True).copy()
N_FOLDS   = 5
TEST_FRAC = 0.20

# Stratify on IDH; patients with unknown IDH form a separate stratum (-1)
idh_strat = cohort['idh'].fillna(-1).astype(int)

sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED)
train_idx, test_idx = next(sss.split(cohort, idh_strat))

cohort['split'] = 'train'
cohort.loc[test_idx, 'split'] = 'test'
cohort['fold'] = -1

train_mask = cohort['split']=='train'
train_df   = cohort[train_mask]
idh_strat_train = train_df['idh'].fillna(-1).astype(int)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold_i, (_, val_idx) in enumerate(skf.split(train_df, idh_strat_train)):
    global_idx = train_df.iloc[val_idx].index
    cohort.loc[global_idx, 'fold'] = fold_i

n_train = (cohort['split']=='train').sum()
n_test  = (cohort['split']=='test').sum()

assert cohort['patient_id'].is_unique, 'Duplicate patient_id — split integrity failed!'
print(f'Train: {n_train}  Test: {n_test}  Total: {len(cohort)}')
print('Fold distribution (train):')
print(cohort[train_mask]['fold'].value_counts().sort_index())
print('\nNo duplicate patient_ids — split integrity verified.')

In [ ]:
print('Class balance per marker and split:')
for col in ['idh','codeletion_1p19q','mgmt','who_grade']:
    if col not in cohort:
        continue
    print(f'\n── {col} ──')
    tbl = cohort.groupby('split')[col].value_counts(dropna=False).unstack(fill_value=0)
    print(tbl)

## 9. Volume path helpers

UCSF-PDGM volume layout:
```
{VOL_ROOT}/{patient_id}_nifti/{patient_id}_{SEQ}.nii/{patient_id}_{SEQ}.nii
```
Sequences: `T1`, `T1c`, `T2`, `FLAIR`, `tumor_segmentation`

In [ ]:
SEQ_MAP = {
    't1'  : 'T1',
    't1ce': 'T1c',
    't2'  : 'T2',
    'flair': 'FLAIR',
}
SEQ_KEYS = list(SEQ_MAP.keys())

# UCSF-PDGM segmentation labels (same as BraTS — verified against TCIA documentation)
REGION_LABELS = {'et': 4, 'ncr': 1, 'ed': 2}

def _vol_path(patient_id, seq_key):
    """Return Path to NIfTI volume or None if not found."""
    seq_name = SEQ_MAP[seq_key]
    nifti_dir = VOL_ROOT / f'{patient_id}_nifti'
    # Layout: {pid}_T1.nii/{pid}_T1.nii  (directory wrapping the file)
    candidate = nifti_dir / f'{patient_id}_{seq_name}.nii' / f'{patient_id}_{seq_name}.nii'
    if candidate.exists():
        return candidate
    # Some patients may have .nii.gz
    gz = candidate.with_suffix('.nii.gz')
    return gz if gz.exists() else None

def _seg_path(patient_id):
    nifti_dir = VOL_ROOT / f'{patient_id}_nifti'
    candidate = (
        nifti_dir
        / f'{patient_id}_tumor_segmentation.nii'
        / f'{patient_id}_tumor_segmentation.nii'
    )
    if candidate.exists():
        return candidate
    gz = candidate.with_suffix('.nii.gz')
    return gz if gz.exists() else None

def _load_sitk(path):
    try:
        return sitk.ReadImage(str(path))
    except Exception:
        return None

def _make_mask(seg_img, label):
    arr = sitk.GetArrayFromImage(seg_img)
    out = sitk.GetImageFromArray((arr==label).astype(np.uint8))
    out.CopyInformation(seg_img)
    return out

# Quick sanity check on a single patient
sample_pid = cohort['patient_id'].iloc[0]
print(f'Sample patient: {sample_pid}')
for sk in SEQ_KEYS:
    p = _vol_path(sample_pid, sk)
    print(f'  {sk:6s}: {p}')
print(f'  seg  : {_seg_path(sample_pid)}')

## 10. Interpretable tumor descriptors

In [ ]:
def compute_descriptors(seg_img):
    """Compute interpretable tumor descriptors from the segmentation.

    All volumes use the actual voxel spacing (not assumed isotropic).
    Division-by-zero → NaN + flag. Never crashes or fills with 0.
    """
    arr  = sitk.GetArrayFromImage(seg_img)
    sp   = seg_img.GetSpacing()   # (x, y, z) mm
    vvol = float(sp[0]*sp[1]*sp[2])

    n_et  = int((arr==4).sum())
    n_ncr = int((arr==1).sum())
    n_ed  = int((arr==2).sum())
    n_tot = n_et + n_ncr + n_ed

    d = {
        'missing__et' : int(n_et  == 0),
        'missing__ncr': int(n_ncr == 0),
        'missing__ed' : int(n_ed  == 0),
    }

    d['desc__total_volume_mm3'] = n_tot*vvol if n_tot>0 else np.nan

    if n_tot > 0:
        d['desc__enhancing_fraction'] = n_et  / n_tot
        d['desc__necrosis_fraction']  = n_ncr / n_tot
    else:
        d['desc__enhancing_fraction'] = np.nan
        d['desc__necrosis_fraction']  = np.nan

    n_core = n_et + n_ncr
    d['desc__edema_to_core_ratio'] = (n_ed/n_core) if n_core>0 else np.nan

    return d

## 11. Pyradiomics extractor setup

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Feature extraction — implemented directly with numpy/scipy/skimage
# to avoid pyradiomics Python 3.13 incompatibility.
#
# Feature names follow the VBX naming convention so Layer 2 SHAP
# grouping and contracts.py validation work without any changes:
#   {seq}__{region}__firstorder__{feat}
#   {seq}__{region}__glcm__{feat}
#   shape__{region}__shape__{feat}
# ═══════════════════════════════════════════════════════════════════

def _firstorder(voxels):
    """Return dict of first-order statistics for a 1-D voxel array."""
    if len(voxels) == 0:
        return {}
    v = voxels.astype(float)
    p10, p25, p75, p90 = np.percentile(v, [10, 25, 75, 90])
    hist, _ = np.histogram(v, bins=64)
    hist_prob = hist / (hist.sum() + 1e-10)
    entropy = float(-np.sum(hist_prob * np.log2(hist_prob + 1e-10)))
    return {
        'Mean'                 : float(np.mean(v)),
        'Variance'             : float(np.var(v)),
        'Skewness'             : float(stats.skew(v)),
        'Kurtosis'             : float(stats.kurtosis(v)),
        'Minimum'              : float(np.min(v)),
        'Maximum'              : float(np.max(v)),
        '10Percentile'         : float(p10),
        '90Percentile'         : float(p90),
        'InterquartileRange'   : float(p75 - p25),
        'Median'               : float(np.median(v)),
        'MeanAbsoluteDeviation': float(np.mean(np.abs(v - np.mean(v)))),
        'RootMeanSquared'      : float(np.sqrt(np.mean(v**2))),
        'Energy'               : float(np.sum(v**2)),
        'Entropy'              : entropy,
        'Range'                : float(np.max(v) - np.min(v)),
    }


def _glcm_features(vol_arr, mask_arr):
    """Return GLCM texture features averaged over axial slices with content.

    Uses 4 in-plane offsets (0°, 45°, 90°, 135°). Averages across slices
    that have ≥16 masked voxels.
    """
    # Quantise to 32 grey levels inside the mask
    masked = vol_arr[mask_arr > 0].astype(float)
    if len(masked) < 16:
        return {}
    vmin, vmax = masked.min(), masked.max()
    if vmax == vmin:
        return {}

    def _quantise(arr, levels=32):
        arr = np.clip(arr, vmin, vmax)
        return ((arr - vmin) / (vmax - vmin) * (levels - 1)).astype(np.uint8)

    offsets = [(0,1), (1,1), (1,0), (1,-1)]   # 0°, 45°, 90°, 135°
    props   = ['contrast','correlation','energy','homogeneity','dissimilarity']
    accum   = {p: [] for p in props}

    for z in range(vol_arr.shape[0]):
        sl_mask = mask_arr[z]
        if sl_mask.sum() < 16:
            continue
        sl_img = _quantise(vol_arr[z])
        sl_img[sl_mask == 0] = 0     # zero out non-mask voxels
        try:
            glcm = graycomatrix(
                sl_img, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                levels=32, symmetric=True, normed=True
            )
            for p in props:
                accum[p].append(float(graycoprops(glcm, p).mean()))
        except Exception:
            continue

    if not accum['contrast']:
        return {}

    # Capitalise to match pyradiomics naming
    name_map = {
        'contrast'     : 'Contrast',
        'correlation'  : 'Correlation',
        'energy'       : 'Energy',
        'homogeneity'  : 'Homogeneity',
        'dissimilarity': 'Dissimilarity',
    }
    return {name_map[p]: float(np.mean(accum[p])) for p in props}


def _shape_features(mask_arr, spacing_xyz):
    """Return 3D shape features from a binary mask.

    spacing_xyz: (x, y, z) voxel size in mm (SimpleITK order).
    Converted to (z, y, x) for numpy array order.
    """
    if mask_arr.sum() < 8:
        return {}

    sp = (spacing_xyz[2], spacing_xyz[1], spacing_xyz[0])  # numpy (z,y,x)
    voxel_vol = float(sp[0] * sp[1] * sp[2])
    n_voxels  = int(mask_arr.sum())
    vol_mm3   = n_voxels * voxel_vol

    feats = {'VoxelVolume': vol_mm3}

    try:
        # Surface area via marching cubes
        verts, faces, _, _ = measure.marching_cubes(
            mask_arr.astype(float), level=0.5, spacing=sp
        )
        # Triangle area sum
        tris = verts[faces]
        cross = np.cross(tris[:,1]-tris[:,0], tris[:,2]-tris[:,0])
        sa = float(0.5 * np.linalg.norm(cross, axis=1).sum())
        feats['MeshSurfaceArea'] = sa

        # Sphericity = (pi^(1/3) * (6V)^(2/3)) / A
        if sa > 0:
            feats['Sphericity'] = float(
                np.pi**(1/3) * (6 * vol_mm3)**(2/3) / sa
            )
    except Exception:
        pass

    try:
        # Axis lengths from inertia tensor eigenvalues
        props = measure.regionprops(
            mask_arr.astype(np.uint8), spacing=sp
        )
        if props:
            p = props[0]
            feats['MajorAxisLength'] = float(p.axis_major_length)
            feats['MinorAxisLength'] = float(p.axis_minor_length)
            if hasattr(p, 'inertia_tensor_eigvals'):
                eigs = sorted(p.inertia_tensor_eigvals, reverse=True)
                if eigs[0] > 0:
                    feats['Elongation'] = float(np.sqrt(eigs[1] / eigs[0]))
                if eigs[0] > 0:
                    feats['Flatness']   = float(np.sqrt(eigs[2] / eigs[0]))
    except Exception:
        pass

    return feats


print('Feature functions defined')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Feature extraction — implemented directly with numpy/scipy/skimage
# to avoid pyradiomics Python 3.13 incompatibility.
#
# Feature names follow the VBX naming convention so Layer 2 SHAP
# grouping and contracts.py validation work without any changes:
#   {seq}__{region}__firstorder__{feat}
#   {seq}__{region}__glcm__{feat}
#   shape__{region}__shape__{feat}
# ═══════════════════════════════════════════════════════════════════

def _firstorder(voxels):
    """Return dict of first-order statistics for a 1-D voxel array."""
    if len(voxels) == 0:
        return {}
    v = voxels.astype(float)
    p10, p25, p75, p90 = np.percentile(v, [10, 25, 75, 90])
    hist, _ = np.histogram(v, bins=64)
    hist_prob = hist / (hist.sum() + 1e-10)
    entropy = float(-np.sum(hist_prob * np.log2(hist_prob + 1e-10)))
    return {
        'Mean'                 : float(np.mean(v)),
        'Variance'             : float(np.var(v)),
        'Skewness'             : float(stats.skew(v)),
        'Kurtosis'             : float(stats.kurtosis(v)),
        'Minimum'              : float(np.min(v)),
        'Maximum'              : float(np.max(v)),
        '10Percentile'         : float(p10),
        '90Percentile'         : float(p90),
        'InterquartileRange'   : float(p75 - p25),
        'Median'               : float(np.median(v)),
        'MeanAbsoluteDeviation': float(np.mean(np.abs(v - np.mean(v)))),
        'RootMeanSquared'      : float(np.sqrt(np.mean(v**2))),
        'Energy'               : float(np.sum(v**2)),
        'Entropy'              : entropy,
        'Range'                : float(np.max(v) - np.min(v)),
    }


def _glcm_features(vol_arr, mask_arr):
    """Return GLCM texture features averaged over axial slices with content.

    Uses 4 in-plane offsets (0°, 45°, 90°, 135°). Averages across slices
    that have ≥16 masked voxels.
    """
    # Quantise to 32 grey levels inside the mask
    masked = vol_arr[mask_arr > 0].astype(float)
    if len(masked) < 16:
        return {}
    vmin, vmax = masked.min(), masked.max()
    if vmax == vmin:
        return {}

    def _quantise(arr, levels=32):
        arr = np.clip(arr, vmin, vmax)
        return ((arr - vmin) / (vmax - vmin) * (levels - 1)).astype(np.uint8)

    offsets = [(0,1), (1,1), (1,0), (1,-1)]   # 0°, 45°, 90°, 135°
    props   = ['contrast','correlation','energy','homogeneity','dissimilarity']
    accum   = {p: [] for p in props}

    for z in range(vol_arr.shape[0]):
        sl_mask = mask_arr[z]
        if sl_mask.sum() < 16:
            continue
        sl_img = _quantise(vol_arr[z])
        sl_img[sl_mask == 0] = 0     # zero out non-mask voxels
        try:
            glcm = graycomatrix(
                sl_img, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                levels=32, symmetric=True, normed=True
            )
            for p in props:
                accum[p].append(float(graycoprops(glcm, p).mean()))
        except Exception:
            continue

    if not accum['contrast']:
        return {}

    # Capitalise to match pyradiomics naming
    name_map = {
        'contrast'     : 'Contrast',
        'correlation'  : 'Correlation',
        'energy'       : 'Energy',
        'homogeneity'  : 'Homogeneity',
        'dissimilarity': 'Dissimilarity',
    }
    return {name_map[p]: float(np.mean(accum[p])) for p in props}


def _shape_features(mask_arr, spacing_xyz):
    """Return 3D shape features from a binary mask.

    spacing_xyz: (x, y, z) voxel size in mm (SimpleITK order).
    Converted to (z, y, x) for numpy array order.
    """
    if mask_arr.sum() < 8:
        return {}

    sp = (spacing_xyz[2], spacing_xyz[1], spacing_xyz[0])  # numpy (z,y,x)
    voxel_vol = float(sp[0] * sp[1] * sp[2])
    n_voxels  = int(mask_arr.sum())
    vol_mm3   = n_voxels * voxel_vol

    feats = {'VoxelVolume': vol_mm3}

    try:
        # Surface area via marching cubes
        verts, faces, _, _ = measure.marching_cubes(
            mask_arr.astype(float), level=0.5, spacing=sp
        )
        # Triangle area sum
        tris = verts[faces]
        cross = np.cross(tris[:,1]-tris[:,0], tris[:,2]-tris[:,0])
        sa = float(0.5 * np.linalg.norm(cross, axis=1).sum())
        feats['MeshSurfaceArea'] = sa

        # Sphericity = (pi^(1/3) * (6V)^(2/3)) / A
        if sa > 0:
            feats['Sphericity'] = float(
                np.pi**(1/3) * (6 * vol_mm3)**(2/3) / sa
            )
    except Exception:
        pass

    try:
        # Axis lengths from inertia tensor eigenvalues
        props = measure.regionprops(
            mask_arr.astype(np.uint8), spacing=sp
        )
        if props:
            p = props[0]
            feats['MajorAxisLength'] = float(p.axis_major_length)
            feats['MinorAxisLength'] = float(p.axis_minor_length)
            if hasattr(p, 'inertia_tensor_eigvals'):
                eigs = sorted(p.inertia_tensor_eigvals, reverse=True)
                if eigs[0] > 0:
                    feats['Elongation'] = float(np.sqrt(eigs[1] / eigs[0]))
                if eigs[0] > 0:
                    feats['Flatness']   = float(np.sqrt(eigs[2] / eigs[0]))
    except Exception:
        pass

    return feats


print('Feature functions defined')


## 12. Run extraction with checkpoint cache

Saves every 25 patients. If the kernel times out, restart and the cache is reloaded.

In [ ]:
CACHE_PATH = OUT_DIR / 'radiomics_cache.pkl'

if CACHE_PATH.exists():
    with open(CACHE_PATH, 'rb') as fh:
        cache = pickle.load(fh)
    print(f'Cache loaded: {len(cache)} patients')
else:
    cache = {}

remaining = cohort[~cohort['patient_id'].isin(cache)]
print(f'Remaining: {len(remaining)} patients')

for i, (_, pat) in enumerate(tqdm(remaining.iterrows(), total=len(remaining))):
    pid = pat['patient_id']
    cache[pid] = extract_patient(pid)
    if (i+1) % 25 == 0:
        with open(CACHE_PATH, 'wb') as fh:
            pickle.dump(cache, fh)
        print(f'  checkpoint: {i+1} done')

with open(CACHE_PATH, 'wb') as fh:
    pickle.dump(cache, fh)
print(f'Extraction complete. Total: {len(cache)}')

## 13. Assemble feature table

In [ ]:
rows = []
for _, pat in cohort.iterrows():
    pid = pat['patient_id']
    feat = cache.get(pid, {})
    merged = pat.to_dict()
    merged.update(feat)
    rows.append(merged)

features_df = pd.DataFrame(rows)
print(f'Feature table: {features_df.shape[0]} rows × {features_df.shape[1]} cols')

# NaN summary
nan_counts = features_df.isna().sum()
print(f'Columns with any NaN: {(nan_counts>0).sum()}')

print('\nMissing labels:')
for col in ['idh','codeletion_1p19q','mgmt','who_grade']:
    if col in features_df:
        n = features_df[col].isna().sum()
        print(f'  {col}: {n} / {len(features_df)}')

print('\nRegion flags:')
for col in [c for c in features_df.columns if c.startswith('missing__')]:
    print(f'  {col}: {int(features_df[col].sum())} patients')

## 14. Pandera schema validation

In [ ]:
_ALLOWED_PREFIXES = (
    't1__','t1ce__','t2__','flair__',
    'shape__','desc__','missing__',
    'age','sex','patient_id','split','fold',
    'idh','codeletion_1p19q','mgmt','who_grade',
)
_LEAKAGE_KEYWORDS = (
    'resection','treatment','survival','vital',
    'mgmt_index','post_op','diagnosis',
)

def _no_leakage(df):
    return all(
        not any(lk in c.lower() for lk in _LEAKAGE_KEYWORDS)
        for c in df.columns
    )

def _allowed_prefix(df):
    return all(
        any(c.startswith(p) for p in _ALLOWED_PREFIXES)
        for c in df.columns
    )

SCHEMA = DataFrameSchema(
    columns={
        'patient_id'         : Column(str, nullable=False),
        'split'              : Column(str, Check.isin(['train','test']), nullable=False),
        'fold'               : Column(pa.Int64, Check.ge(-1), nullable=False),
        'age'                : Column(pa.Float64, [Check.ge(18), Check.le(100)], nullable=False),
        'sex'                : Column(pa.Int64, Check.isin([0,1]), nullable=False),
        'idh'                : Column(pa.Float64, Check.isin([0.,1.]), nullable=True),
        'codeletion_1p19q'   : Column(pa.Float64, Check.isin([0.,1.]), nullable=True),
        'mgmt'               : Column(pa.Float64, Check.isin([0.,1.]), nullable=True),
        'who_grade'          : Column(pa.Float64, Check.isin([2.,3.,4.]), nullable=True),
        'desc__total_volume_mm3'     : Column(pa.Float64, Check.ge(0), nullable=True),
        'desc__enhancing_fraction'   : Column(pa.Float64, [Check.ge(0),Check.le(1)], nullable=True),
        'desc__necrosis_fraction'    : Column(pa.Float64, [Check.ge(0),Check.le(1)], nullable=True),
        'desc__edema_to_core_ratio'  : Column(pa.Float64, Check.ge(0), nullable=True),
    },
    checks=[
        Check(_no_leakage,     error='Leakage column detected'),
        Check(_allowed_prefix, error='Column with disallowed prefix'),
        Check(lambda df: df['patient_id'].is_unique, error='Duplicate patient_id'),
    ],
    strict=False,
)

features_df['fold'] = features_df['fold'].astype('Int64')
features_df['sex']  = features_df['sex'].astype('Int64')

try:
    validated = SCHEMA.validate(features_df, lazy=True)
    print('✓ Pandera schema validation PASSED')
    print(f'  {len(validated)} rows × {len(validated.columns)} columns')
except pa.errors.SchemaErrors as e:
    print('✗ Schema FAILED:')
    print(e.failure_cases)
    raise

## 15. Hero demo cases

In [ ]:
test_df = features_df[features_df['split']=='test'].copy()
fully_labelled = test_df.dropna(subset=['idh','codeletion_1p19q','mgmt'])

hero_ids = []
for idh_v in [1.,0.]:
    for mgmt_v in [1.,0.]:
        sub = fully_labelled[
            (fully_labelled['idh']==idh_v) & (fully_labelled['mgmt']==mgmt_v)
        ]
        hero_ids.extend(sub['patient_id'].iloc[:2].tolist())

hero_ids = hero_ids[:8]
print(f'Hero cases ({len(hero_ids)}):')
for pid in hero_ids:
    r = test_df[test_df['patient_id']==pid].iloc[0]
    print(
        f'  {pid}  IDH={r.get("idh","?"):.0f}  '
        f'1p19q={r.get("codeletion_1p19q","?"):.0f}  '
        f'MGMT={r.get("mgmt","?"):.0f}  '
        f'WHO={r.get("who_grade","?")}'
    )
print('\nNote: volumes NOT committed pending BraTS/UCSF-PDGM license redistribution check with Tanish.')

## 16. Export and sha256

In [ ]:
OUT_PARQUET = OUT_DIR / 'features.parquet'
features_df.to_parquet(OUT_PARQUET, index=False)

sha256 = hashlib.sha256(OUT_PARQUET.read_bytes()).hexdigest()
print(f'features.parquet: {OUT_PARQUET.stat().st_size/1024:.1f} KB')
print(f'sha256          : {sha256}')

provenance.update({
    'feature_table_sha256': sha256,
    'feature_table_rows'  : len(features_df),
    'feature_table_cols'  : len(features_df.columns),
    'n_train'             : n_train,
    'n_test'              : n_test,
    'n_raw_csv'           : n_before,
    'n_followup_dropped'  : n_dropped,
    'n_after_dedup'       : n_after,
    'n_missing_vol'       : n_missing_vol,
})

(OUT_DIR/'provenance.json').write_text(json.dumps(provenance, indent=2))
print('\nFull provenance:')
print(json.dumps(provenance, indent=2))

## 17. Exclusion table

In [ ]:
print('='*62)
print('EXCLUSION TABLE')
print('='*62)
rows_tbl = [
    ('Rows in raw UCSF-PDGM metadata CSV',     n_before,      '—'),
    ('Follow-up rows dropped (dedup)',          n_dropped,     'Keep baseline; _FU suffix = post-op scan'),
    ('After dedup',                             n_after,       '—'),
    ('Missing volume directory on disk',        n_missing_vol, 'Patient dir not in filtered dataset'),
    ('Final cohort',                            n_matched,     '—'),
    ('Train split',                             n_train,       'Stratified on IDH, seed=42, 80%'),
    ('Test split',                              n_test,        'Stratified on IDH, seed=42, 20%'),
]
for label, count, reason in rows_tbl:
    print(f'  {label:<45} {count:>5}  {reason}')

print('\nMissing labels (targets — never imputed):')
for col in ['idh','codeletion_1p19q','mgmt','who_grade']:
    if col in features_df:
        n = features_df[col].isna().sum()
        print(f'  {col:<25} {n:>4} ({100*n/len(features_df):.1f}%)')